# Stage 1: Data Cleaning
Reads six raw tables from S3, applies per table cleaning rules, writes cleaned CSVs.

Input: s3://ins-churn-data/*.csv (bucket root)

Output: s3://ins-churn-data/cleaned/*.csv

In [ ]:
import boto3
import pandas as pd
import io

BUCKET_NAME = 'ins-churn-data'
REGION = 'us-east-1'
s3 = boto3.client('s3', region_name=REGION)

# List every object, confirms auth works and shows exact key paths
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=BUCKET_NAME)
all_keys = []
for page in pages:
    for obj in page.get('Contents', []):
        all_keys.append(obj['Key'])
        print(f"  {obj['Key']:<60} {obj['Size']:>10,} bytes")
print(f'Total: {len(all_keys)} objects')

In [ ]:
import boto3, pandas as pd, numpy as np, io, os, json, logging, pickle

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

BUCKET_NAME = 'ins-churn-data'
REGION = 'us-east-1'
s3 = boto3.client('s3', region_name=REGION)
LOCAL_TMP = '/tmp/ins_churn'  # only writable local path in SageMaker Notebook
os.makedirs(LOCAL_TMP, exist_ok=True)


def read_csv_from_s3(bucket, key, **kwargs):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(response['Body'].read()), **kwargs)


def write_csv_to_s3(df, bucket, key):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    log.info('  saved to s3://%s/%s (%d rows)', bucket, key, len(df))


def write_json_to_s3(data, bucket, key):
    s3.put_object(Bucket=bucket, Key=key,
                  Body=json.dumps(data, indent=2, default=str))


def upload_file_to_s3(local_path, bucket, key):
    s3.upload_file(local_path, bucket, key)  # for model artifacts via /tmp


def make_key(folder, filename):
    return f'{folder}/{filename}' if folder else filename


log.info('Helpers ready.')

In [ ]:
INPUT_DIR = ''  # raw files at bucket root
OUTPUT_DIR = 'cleaned'


def clean_products(df):
    for col in ['unit_price', 'cost']:  # stored as '1,248' strings
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '', regex=False), errors='coerce'
            )
    if 'active_flag' in df.columns:  # mixed: True/'08/08/2018'/False
        df['active_flag'] = df['active_flag'].astype(str).str.strip().str.lower()\
            .map({'true': 1, 'false': 0})
    return df


def clean_customers(df):
    if 'status' in df.columns:  # mixed: 'Active'/'ACTIVE'/'active'
        df['status'] = df['status'].astype(str).str.strip().str.lower()
    return df


def clean_fact(df):
    for col in ['premium_amount', 'transaction_amount']:  # stored as '3,220'
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '', regex=False), errors='coerce'
            )
    return df


def clean_agents(df):
    if 'active' in df.columns:
        df['active'] = df['active'].astype(str).str.lower().map({'true': 1, 'false': 0})
    return df


def clean_geography(df):
    return df


def clean_bridge(df):
    return df


tables = {
    'dim_agents': (make_key(INPUT_DIR, 'dim_agents.csv'), clean_agents),
    'dim_customers': (make_key(INPUT_DIR, 'dim_customers.csv'), clean_customers),
    'dim_products': (make_key(INPUT_DIR, 'dim_products.csv'), clean_products),
    'dim_geography': (make_key(INPUT_DIR, 'dim_geography.csv'), clean_geography),
    'bridge_customer_agent': (make_key(INPUT_DIR, 'bridge_customer_agent.csv'), clean_bridge),
    'fact_policy_activity': (make_key(INPUT_DIR, 'fact_policy_activity.csv'), clean_fact),
}

for name, (key, cleaner) in tables.items():
    df = read_csv_from_s3(BUCKET_NAME, key)
    df_clean = cleaner(df)
    write_csv_to_s3(df_clean, BUCKET_NAME, make_key(OUTPUT_DIR, f'{name}_cleaned.csv'))
    log.info('  %s produced %d rows', name, len(df_clean))